In [1]:
# ============================================================
# CONNECT DRIVE + LOAD GTSRB DATASET
# ============================================================

from google.colab import drive
import os
import zipfile
import pandas as pd
import numpy as np
import joblib

from PIL import Image
from skimage.feature import hog
from sklearn.svm import SVC

# Connect Google Drive
drive.mount("/content/drive")

# ZIP location in Google Drive
zip_path = "/content/drive/MyDrive/TrafficSignProject/data.zip"

# Dataset location for this Colab runtime
dataset_path = "/content/traffic_signs"

# Extract dataset
if not os.path.exists(os.path.join(dataset_path, "Train.csv")):

    print("Extracting GTSRB dataset...")

    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall(dataset_path)

    print("Dataset extracted successfully.")

else:
    print("GTSRB dataset already loaded.")

print("Dataset files:")
print(os.listdir(dataset_path))

Mounted at /content/drive
Extracting GTSRB dataset...
Dataset extracted successfully.
Dataset files:
['Meta.csv', 'signnames.csv', 'Train', 'Test', 'Meta', 'Train.csv', 'Test.csv']


In [2]:
# ============================================================
# TRAIN HOG + SVM CLASSIFIER
# ============================================================

from sklearn.metrics import accuracy_score, classification_report


# ------------------------------------------------------------
# 1. Load GTSRB training data
# ------------------------------------------------------------

train_df = pd.read_csv(
    f"{dataset_path}/Train.csv"
)

print("Training images:", len(train_df))


# ------------------------------------------------------------
# 2. HOG feature extraction
# ------------------------------------------------------------

def extract_hog_features(image_path):

    image = Image.open(image_path).convert("RGB")

    # Resize every image to the same size
    image = image.resize((32, 32))

    image = np.array(image)

    # Convert RGB to grayscale
    gray = np.mean(
        image,
        axis=2
    ).astype(np.uint8)

    # Extract HOG features
    features = hog(
        gray,
        orientations=9,
        pixels_per_cell=(8, 8),
        cells_per_block=(2, 2)
    )

    return features


# ------------------------------------------------------------
# 3. Create feature matrix and labels
# ------------------------------------------------------------

X_train = []
y_train = []

for _, row in train_df.iterrows():

    image_path = row["Path"].replace(
        "/code",
        dataset_path
    )

    features = extract_hog_features(
        image_path
    )

    X_train.append(features)
    y_train.append(row["ClassId"])


X_train = np.array(X_train)
y_train = np.array(y_train)

print("HOG feature matrix:", X_train.shape)
print("Training labels:", y_train.shape)


# ------------------------------------------------------------
# 4. Train SVM classifier
# ------------------------------------------------------------

svm_model = SVC(
    kernel="rbf",
    C=10,
    gamma="scale"
)

svm_model.fit(
    X_train,
    y_train
)

print("SVM training completed.")


# ------------------------------------------------------------
# 5. Load separate GTSRB test data
# ------------------------------------------------------------

test_df = pd.read_csv(
    f"{dataset_path}/Test.csv"
)

print("Test images:", len(test_df))


# ------------------------------------------------------------
# 6. Extract HOG features from test images
# ------------------------------------------------------------

X_test = []
y_test = []

for _, row in test_df.iterrows():

    image_path = row["Path"].replace(
        "/code",
        dataset_path
    )

    features = extract_hog_features(
        image_path
    )

    X_test.append(features)
    y_test.append(row["ClassId"])


X_test = np.array(X_test)
y_test = np.array(y_test)


# ------------------------------------------------------------
# 7. Predict test classes
# ------------------------------------------------------------

y_pred = svm_model.predict(
    X_test
)


# ------------------------------------------------------------
# 8. Evaluate the classifier
# ------------------------------------------------------------

accuracy = accuracy_score(
    y_test,
    y_pred
)

print(
    f"Test Accuracy: {accuracy * 100:.2f}%"
)

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        y_pred
    )
)


# ------------------------------------------------------------
# 9. Save the trained SVM model
# ------------------------------------------------------------

SVM_MODEL_PATH = (
    "/content/drive/MyDrive/"
    "TrafficSignProject/"
    "traffic_sign_final_model.pkl"
)

joblib.dump(
    svm_model,
    SVM_MODEL_PATH
)

print("\nSVM model saved successfully.")
print(SVM_MODEL_PATH)

Training images: 39209
HOG feature matrix: (39209, 324)
Training labels: (39209,)
SVM training completed.
Test images: 12630
Test Accuracy: 83.76%

Classification Report:
              precision    recall  f1-score   support

           0       0.79      0.62      0.69        60
           1       0.72      0.76      0.74       720
           2       0.74      0.77      0.75       750
           3       0.59      0.69      0.63       450
           4       0.91      0.93      0.92       660
           5       0.62      0.75      0.68       630
           6       0.87      0.77      0.82       150
           7       0.83      0.75      0.79       450
           8       0.75      0.69      0.72       450
           9       0.91      0.84      0.87       480
          10       0.95      0.95      0.95       660
          11       0.66      0.84      0.74       420
          12       0.97      0.99      0.98       690
          13       0.99      1.00      1.00       720
          14      

In [3]:
# Driver assistance rules

driver_actions = {
    "Stop": "Brake and bring the vehicle to a complete stop.",
    "Yield": "Slow down and prepare to give way.",
    "Speed limit (20km/h)": "Reduce speed to 20 km/h or below.",
    "Speed limit (30km/h)": "Reduce speed to 30 km/h or below.",
    "Speed limit (50km/h)": "Reduce speed to 50 km/h or below.",
    "Speed limit (60km/h)": "Reduce speed to 60 km/h or below.",
    "Speed limit (70km/h)": "Reduce speed to 70 km/h or below.",
    "Speed limit (80km/h)": "Reduce speed to 80 km/h or below.",
    "Speed limit (100km/h)": "Reduce speed to 100 km/h or below.",
    "Speed limit (120km/h)": "Reduce speed to 120 km/h or below.",
    "Turn right ahead": "Right turn ahead. Driver decision required.",
    "Turn left ahead": "Left turn ahead. Driver decision required.",
    "Dangerous curve to the right": "Slow down and prepare for a right curve.",
    "Dangerous curve to the left": "Slow down and prepare for a left curve.",
    "Road work": "Warning: Road work ahead. Reduce speed.",
    "Pedestrians": "Watch for pedestrians and reduce speed.",
    "Children crossing": "Slow down and watch for children.",
    "Bicycles crossing": "Watch for bicycles and reduce speed.",
    "Traffic signals": "Prepare for traffic signals ahead.",
    "No entry": "Warning: Do not enter this road.",
    "Keep right": "Keep to the right side of the road.",
    "Keep left": "Keep to the left side of the road.",
    "Roundabout mandatory": "Prepare for the roundabout."
}

joblib.dump(driver_actions, "/content/driver_actions.pkl")

print("driver_actions.pkl saved successfully!")

driver_actions.pkl saved successfully!


In [4]:
# Save sign categories

sign_categories = {
    0: "Speed Limit",
    1: "Speed Limit",
    2: "Speed Limit",
    3: "Speed Limit",
    4: "Speed Limit",
    5: "Speed Limit",
    6: "Speed Limit",
    7: "Speed Limit",
    8: "Speed Limit",
    9: "Prohibitory",
    10: "Prohibitory",
    11: "Priority",
    12: "Priority",
    13: "Priority",
    14: "Prohibitory",
    15: "Prohibitory",
    16: "Prohibitory",
    17: "Prohibitory",
    18: "Warning",
    19: "Warning",
    20: "Warning",
    21: "Warning",
    22: "Warning",
    23: "Warning",
    24: "Warning",
    25: "Warning",
    26: "Warning",
    27: "Warning",
    28: "Warning",
    29: "Warning",
    30: "Warning",
    31: "Warning",
    32: "End",
    33: "Mandatory",
    34: "Mandatory",
    35: "Mandatory",
    36: "Mandatory",
    37: "Mandatory",
    38: "Mandatory",
    39: "Mandatory",
    40: "Mandatory",
    41: "End",
    42: "End"
}

joblib.dump(sign_categories, "/content/sign_categories.pkl")

print("sign_categories.pkl saved successfully!")

sign_categories.pkl saved successfully!


In [5]:
import os

for root, dirs, files in os.walk("/content/drive/MyDrive"):
    for file in files:
        if file == "traffic_sign_final_model.pkl":
            print(os.path.join(root, file))

/content/drive/MyDrive/TrafficSignProject/traffic_sign_final_model.pkl


In [6]:
# =========================================================
# TRAFFIC SIGN RECOGNITION & DRIVER ASSISTANCE SIMULATION
# Multiple scenarios + adjustable starting speed
# =========================================================

from IPython.display import HTML, display
import base64
from pathlib import Path
import json

# Real traffic-sign images from the GTSRB dataset
sign_paths = {
    14: "/content/traffic_signs/Train/14/00014_00000_00000.png",   # STOP
    2:  "/content/traffic_signs/Train/2/00002_00000_00000.png",    # Speed 50
    20: "/content/traffic_signs/Train/20/00020_00000_00000.png",  # Curve right
    13: "/content/traffic_signs/Train/13/00013_00000_00000.png",  # Yield
    25: "/content/traffic_signs/Train/25/00025_00000_00000.png",  # Road work
    33: "/content/traffic_signs/Train/33/00033_00000_00000.png",  # Turn right
    34: "/content/traffic_signs/Train/34/00034_00000_00000.png"   # Turn left
}

sign_images = {}

for class_id, path in sign_paths.items():
    try:
        encoded = base64.b64encode(Path(path).read_bytes()).decode()
        sign_images[class_id] = f"data:image/png;base64,{encoded}"
    except:
        sign_images[class_id] = ""

# Each sign has its own driving response
scenarios = {
    14: {
        "name": "STOP",
        "action": "Vehicle slows down and stops",
        "target": 0,
        "behavior": "stop"
    },
    2: {
        "name": "Speed Limit 50",
        "action": "Vehicle reduces speed to 50 km/h",
        "target": 50,
        "behavior": "slow"
    },
    20: {
        "name": "Dangerous Curve Right",
        "action": "Vehicle slows and follows the right curve",
        "target": 35,
        "behavior": "curve-right"
    },
    13: {
        "name": "YIELD",
        "action": "Vehicle slows and continues carefully",
        "target": 25,
        "behavior": "yield"
    },
    25: {
        "name": "Road Work",
        "action": "Vehicle slows and proceeds carefully",
        "target": 30,
        "behavior": "slow"
    },
    33: {
        "name": "Turn Right Ahead",
        "action": "Vehicle slows and takes the right turn",
        "target": 30,
        "behavior": "turn-right"
    },
    34: {
        "name": "Turn Left Ahead",
        "action": "Vehicle slows and takes the left turn",
        "target": 30,
        "behavior": "turn-left"
    }
}

scenario_data = json.dumps(scenarios)
image_data = json.dumps(sign_images)

html = f"""
<!DOCTYPE html>
<html>
<head>

<style>

body {{
    margin: 0;
    font-family: Arial, sans-serif;
    background: #eef1f5;
    color: #111827;
}}

.dashboard {{
    width: 1100px;
    margin: 20px auto;
    background: white;
    border-radius: 18px;
    padding: 20px;
    box-shadow: 0 8px 25px rgba(0,0,0,0.12);
}}

.header {{
    display: flex;
    justify-content: space-between;
    align-items: center;
    margin-bottom: 12px;
}}

.title {{
    font-size: 25px;
    font-weight: bold;
    color: #111827;
}}

.controls {{
    display: flex;
    align-items: center;
    gap: 10px;
}}

select, button {{
    padding: 10px 14px;
    border-radius: 8px;
    border: 1px solid #999;
    font-size: 15px;
    color: #111827;
    background: white;
}}

button {{
    cursor: pointer;
    font-weight: bold;
    background: #f3f4f6;
}}

.speed-control {{
    display: flex;
    align-items: center;
    gap: 10px;
    margin-bottom: 15px;
    color: #111827;
    font-weight: bold;
}}

.speed-control input {{
    width: 230px;
}}

#speedValue {{
    min-width: 70px;
}}

.scene {{
    position: relative;
    height: 430px;
    overflow: hidden;
    border-radius: 14px;
    background: linear-gradient(
        #9ed8ff 0%,
        #dff3ff 58%,
        #78ad62 58%
    );
}}

.road {{
    position: absolute;
    bottom: 0;
    width: 100%;
    height: 165px;
    background: #404348;
}}

.road-line {{
    position: absolute;
    top: 78px;
    width: 100%;
    border-top: 5px dashed #f5f5f5;
}}

.sign-pole {{
    position: absolute;
    right: 150px;
    bottom: 105px;
    width: 8px;
    height: 145px;
    background: #555;
}}

.sign {{
    position: absolute;
    right: 115px;
    bottom: 230px;
    width: 75px;
    height: 75px;
    background: white;
    border-radius: 10px;
    padding: 5px;
    object-fit: contain;
    filter: contrast(2) saturate(1.8) brightness(1.15);
    box-shadow: 0 4px 12px rgba(0,0,0,0.35);
}}

.car {{
    position: absolute;
    left: 40px;
    bottom: 52px;
    width: 120px;
    height: 55px;
    background: #26364a;
    border-radius: 18px 28px 10px 10px;
    box-shadow: 0 7px 10px rgba(0,0,0,0.25);
    transition: bottom 0.7s ease;
}}

.car::before {{
    content: "";
    position: absolute;
    width: 60px;
    height: 30px;
    left: 28px;
    top: -22px;
    background: #344e68;
    border-radius: 20px 20px 5px 5px;
}}

.wheel {{
    position: absolute;
    bottom: -10px;
    width: 25px;
    height: 25px;
    background: #151515;
    border-radius: 50%;
}}

.wheel.left {{
    left: 15px;
}}

.wheel.right {{
    right: 15px;
}}

.info {{
    display: flex;
    gap: 15px;
    margin-top: 15px;
}}

.card {{
    flex: 1;
    padding: 15px;
    border-radius: 12px;
    background: #f1f4f8;
}}

.label {{
    font-size: 13px;
    color: #374151;
}}

.value {{
    margin-top: 5px;
    font-size: 18px;
    font-weight: bold;
    color: #111827;
}}

.status {{
    margin-top: 15px;
    padding: 14px;
    border-radius: 10px;
    text-align: center;
    font-size: 17px;
    font-weight: bold;
    color: #111827;
    background: #e5edff;
}}

</style>
</head>

<body>

<div class="dashboard">

    <div class="header">

        <div class="title">
            Traffic Sign Recognition & Driver Assistance
        </div>

        <div class="controls">

            <select id="scenario">
                <option value="14">STOP</option>
                <option value="2">Speed Limit 50</option>
                <option value="20">Dangerous Curve Right</option>
                <option value="13">YIELD</option>
                <option value="25">Road Work</option>
                <option value="33">Turn Right Ahead</option>
                <option value="34">Turn Left Ahead</option>
            </select>

            <button id="startButton">
                Start Simulation
            </button>

        </div>

    </div>

    <div class="speed-control">

        <span>Initial Speed:</span>

        <input
            type="range"
            id="initialSpeed"
            min="20"
            max="100"
            value="65"
            step="5"
        >

        <span id="speedValue">65 km/h</span>

    </div>

    <div class="scene">

        <div class="road">
            <div class="road-line"></div>
        </div>

        <div class="sign-pole"></div>

        <img id="sign" class="sign">

        <div id="car" class="car">
            <div class="wheel left"></div>
            <div class="wheel right"></div>
        </div>

    </div>

    <div class="info">

        <div class="card">
            <div class="label">Detected Traffic Sign</div>
            <div id="signName" class="value">STOP</div>
        </div>

        <div class="card">
            <div class="label">Current Vehicle Speed</div>
            <div id="speed" class="value">65 km/h</div>
        </div>

        <div class="card">
            <div class="label">Driver Assistance</div>
            <div id="action" class="value">
                Vehicle slows down and stops
            </div>
        </div>

    </div>

    <div id="status" class="status">
        Select a scenario, choose the starting speed and start the simulation.
    </div>

</div>

<script>

const scenarios = {scenario_data};
const images = {image_data};

const scenarioSelect = document.getElementById("scenario");
const speedSlider = document.getElementById("initialSpeed");
const speedValue = document.getElementById("speedValue");
const sign = document.getElementById("sign");
const signName = document.getElementById("signName");
const speedDisplay = document.getElementById("speed");
const action = document.getElementById("action");
const status = document.getElementById("status");
const car = document.getElementById("car");
const startButton = document.getElementById("startButton");

let animationFrame = null;

speedSlider.addEventListener("input", () => {{
    speedValue.innerText = speedSlider.value + " km/h";
}});

function updateScenario() {{

    const id = scenarioSelect.value;
    const data = scenarios[id];

    sign.src = images[id];
    signName.innerText = data.name;
    action.innerText = data.action;
    speedDisplay.innerText = speedSlider.value + " km/h";

}}

scenarioSelect.addEventListener("change", updateScenario);

function startSimulation() {{

    cancelAnimationFrame(animationFrame);

    const id = scenarioSelect.value;
    const data = scenarios[id];

    const initialSpeed = Number(speedSlider.value);
    const targetSpeed = data.target;

    let currentSpeed = initialSpeed;
    let position = 40;
    let lastTime = null;
    let reachedSign = false;

    car.style.left = "40px";
    car.style.bottom = "52px";

    speedDisplay.innerText = initialSpeed + " km/h";
    status.innerText = "Vehicle approaching the traffic sign...";

    function animate(timestamp) {{

        if (!lastTime) {{
            lastTime = timestamp;
        }}

        const elapsed = (timestamp - lastTime) / 1000;
        lastTime = timestamp;

        // Detect the sign when the vehicle gets close to it
        if (position >= 540 && !reachedSign) {{

            reachedSign = true;

            status.innerText =
                data.name + " detected — " + data.action;

        }}

        // Gradually change the vehicle speed after detection
        if (reachedSign) {{

            if (currentSpeed > targetSpeed) {{
                currentSpeed -= 25 * elapsed;

                if (currentSpeed < targetSpeed) {{
                    currentSpeed = targetSpeed;
                }}
            }}

            else if (currentSpeed < targetSpeed) {{
                currentSpeed += 20 * elapsed;

                if (currentSpeed > targetSpeed) {{
                    currentSpeed = targetSpeed;
                }}
            }}

        }}

        speedDisplay.innerText =
            Math.round(currentSpeed) + " km/h";

        // Convert speed into movement across the simulation
        const movement = currentSpeed * elapsed * 3.0;
        position += movement;

        car.style.left = position + "px";

        // STOP: vehicle stops before the traffic sign
        if (data.behavior === "stop" && position >= 700) {{

            currentSpeed = 0;

            speedDisplay.innerText = "0 km/h";
            car.style.left = "700px";

            status.innerText =
                "STOP detected — Vehicle has stopped.";

            return;
        }}

        // Right curve: vehicle changes its path and continues
        if (data.behavior === "curve-right" && position >= 570) {{

            car.style.bottom = "95px";

            status.innerText =
                "Dangerous curve detected — Vehicle slowing and following the right curve.";

        }}

        // Right turn: vehicle moves upward toward the right
        if (data.behavior === "turn-right" && position >= 570) {{

            car.style.bottom = "125px";

            status.innerText =
                "Right turn ahead — Vehicle slowing and taking the right turn.";

        }}

        // Left turn: vehicle moves upward toward the left side
        if (data.behavior === "turn-left" && position >= 570) {{

            car.style.bottom = "125px";

            status.innerText =
                "Left turn ahead — Vehicle slowing and taking the left turn.";

        }}

        // All non-STOP scenarios continue beyond the sign and leave the frame
        if (data.behavior !== "stop" && position > 1150) {{

            speedDisplay.innerText =
                Math.round(currentSpeed) + " km/h";

            status.innerText =
                data.name + " handled — Vehicle continues driving.";

            return;
        }}

        animationFrame = requestAnimationFrame(animate);
    }}

    animationFrame = requestAnimationFrame(animate);
}}

startButton.addEventListener("click", startSimulation);

updateScenario();

</script>

</body>
</html>
"""

# =========================================================
# PUBLIC ENVIRONMENT DEMO
# =========================================================

import gradio as gr


# Put the complete simulation HTML inside an iframe
# so its JavaScript continues to work in the public demo.
import base64

encoded_html = base64.b64encode(
    html.encode("utf-8")
).decode("utf-8")

public_html = f"""
<iframe
    src="data:text/html;base64,{encoded_html}"
    style="
        width:100%;
        height:720px;
        border:none;
        display:block;
        overflow:hidden;
    ">
</iframe>
"""


# Create the public Gradio app
with gr.Blocks(
    title="Traffic Sign Recognition & Driver Assistance"
) as demo:

    gr.HTML(public_html)


# Generate the temporary public URL
demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://2fed5b3a0890ce3682.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [7]:
!pip install -q ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.6/46.6 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 36.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.4/77.4 kB 8.0 MB/s eta 0:00:00


In [8]:
# ============================================================
# LOAD TRAINED YOLO + HOG-SVM MODELS
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

from ultralytics import YOLO
import joblib
import os

# Paths
YOLO_PATH = "/content/drive/MyDrive/TrafficSignProject/traffic_sign_detector.pt"
SVM_PATH = "/content/drive/MyDrive/TrafficSignProject/traffic_sign_final_model.pkl"

# Load models
detector = YOLO(YOLO_PATH)
svm_model = joblib.load(SVM_PATH)

print("YOLO detector loaded:", os.path.exists(YOLO_PATH))
print("SVM classifier loaded:", os.path.exists(SVM_PATH))
print("Both models are ready.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
YOLO detector loaded: True
SVM classifier loaded: True
Both models are ready.


In [9]:
# ============================================================
# FINAL ROAD-SCENE TRAFFIC SIGN ANALYSIS
# YOLO11n + HOG + SVM + Metadata + Scene Intelligence
# ============================================================

import os
import cv2
import joblib
import numpy as np
import pandas as pd
import gradio as gr

from skimage.feature import hog
from ultralytics import YOLO


# ============================================================
# 1. LOAD SAVED MODELS
# ============================================================

YOLO_MODEL_PATH = "/content/drive/MyDrive/TrafficSignProject/traffic_sign_detector.pt"
SVM_MODEL_PATH = "/content/drive/MyDrive/TrafficSignProject/traffic_sign_final_model.pkl"

detector = YOLO(YOLO_MODEL_PATH)
classifier = joblib.load(SVM_MODEL_PATH)

print("YOLO detector loaded successfully.")
print("SVM classifier loaded successfully.")


# ============================================================
# 2. GTSRB SIGN NAMES
# ============================================================

SIGN_NAMES = [
    "Speed limit (20km/h)",
    "Speed limit (30km/h)",
    "Speed limit (50km/h)",
    "Speed limit (60km/h)",
    "Speed limit (70km/h)",
    "Speed limit (80km/h)",
    "End of speed limit (80km/h)",
    "Speed limit (100km/h)",
    "Speed limit (120km/h)",
    "No passing",
    "No passing for vehicles over 3.5 metric tons",
    "Right-of-way at the next intersection",
    "Priority road",
    "Yield",
    "Stop",
    "No vehicles",
    "Vehicles over 3.5 metric tons prohibited",
    "No entry",
    "General caution",
    "Dangerous curve to the left",
    "Dangerous curve to the right",
    "Double curve",
    "Bumpy road",
    "Slippery road",
    "Road narrows on the right",
    "Road work",
    "Traffic signals",
    "Pedestrians",
    "Children crossing",
    "Bicycles crossing",
    "Beware of ice/snow",
    "Wild animals crossing",
    "End of all speed and passing limits",
    "Turn right ahead",
    "Turn left ahead",
    "Ahead only",
    "Go straight or right",
    "Go straight or left",
    "Keep right",
    "Keep left",
    "Roundabout mandatory",
    "End of no passing"
]


# ============================================================
# 3. SIGN CATEGORY
# ============================================================

def get_category(sign_name):

    if "Speed limit" in sign_name:
        return "Regulatory"

    if (
        "No passing" in sign_name
        or "No entry" in sign_name
        or "prohibited" in sign_name
        or sign_name == "No vehicles"
    ):
        return "Prohibitory"

    warning_signs = [
        "General caution",
        "Dangerous curve to the left",
        "Dangerous curve to the right",
        "Double curve",
        "Bumpy road",
        "Slippery road",
        "Road narrows on the right",
        "Road work",
        "Traffic signals",
        "Pedestrians",
        "Children crossing",
        "Bicycles crossing",
        "Beware of ice/snow",
        "Wild animals crossing"
    ]

    if sign_name in warning_signs:
        return "Warning"

    priority_signs = [
        "Right-of-way at the next intersection",
        "Priority road",
        "Yield",
        "Stop"
    ]

    if sign_name in priority_signs:
        return "Priority"

    if "End of" in sign_name:
        return "Derestriction"

    return "Mandatory"


# ============================================================
# 4. DRIVER ASSISTANCE DATABASE
# ============================================================

DRIVER_ACTIONS = {

    "Speed limit (20km/h)": "Maintain speed at or below 20 km/h.",
    "Speed limit (30km/h)": "Maintain speed at or below 30 km/h.",
    "Speed limit (50km/h)": "Maintain speed at or below 50 km/h.",
    "Speed limit (60km/h)": "Maintain speed at or below 60 km/h.",
    "Speed limit (70km/h)": "Maintain speed at or below 70 km/h.",
    "Speed limit (80km/h)": "Maintain speed at or below 80 km/h.",
    "Speed limit (100km/h)": "Maintain speed at or below 100 km/h.",
    "Speed limit (120km/h)": "Maintain speed at or below 120 km/h.",

    "End of speed limit (80km/h)":
        "The previous 80 km/h speed restriction ends.",

    "No passing":
        "Do not overtake other vehicles.",

    "No passing for vehicles over 3.5 metric tons":
        "Heavy vehicles must not overtake.",

    "Right-of-way at the next intersection":
        "Prepare to follow the right-of-way rule at the next intersection.",

    "Priority road":
        "You have priority on the upcoming road.",

    "Yield":
        "Slow down and give way to other road users.",

    "Stop":
        "Stop completely and proceed only when safe.",

    "No vehicles":
        "Do not enter with a vehicle.",

    "Vehicles over 3.5 metric tons prohibited":
        "Heavy vehicles above the specified weight are prohibited.",

    "No entry":
        "Do not enter this road from this direction.",

    "General caution":
        "Proceed carefully and remain alert.",

    "Dangerous curve to the left":
        "Slow down and prepare for a left curve.",

    "Dangerous curve to the right":
        "Slow down and prepare for a right curve.",

    "Double curve":
        "Reduce speed and prepare for successive curves.",

    "Bumpy road":
        "Reduce speed and prepare for uneven road conditions.",

    "Slippery road":
        "Reduce speed and avoid sudden braking or steering.",

    "Road narrows on the right":
        "Reduce speed and prepare for a narrower roadway.",

    "Road work":
        "Slow down and watch for road work and workers.",

    "Traffic signals":
        "Watch for upcoming traffic signals.",

    "Pedestrians":
        "Slow down and watch carefully for pedestrians.",

    "Children crossing":
        "Slow down and watch carefully for children.",

    "Bicycles crossing":
        "Slow down and watch for crossing bicycles.",

    "Beware of ice/snow":
        "Reduce speed and drive cautiously on potentially icy roads.",

    "Wild animals crossing":
        "Reduce speed and watch for animals crossing.",

    "End of all speed and passing limits":
        "Previous speed and passing restrictions end.",

    "Turn right ahead":
        "Prepare to turn right ahead.",

    "Turn left ahead":
        "Prepare to turn left ahead.",

    "Ahead only":
        "Continue straight ahead.",

    "Go straight or right":
        "Continue straight or turn right.",

    "Go straight or left":
        "Continue straight or turn left.",

    "Keep right":
        "Keep to the right side of the road.",

    "Keep left":
        "Keep to the left side of the road.",

    "Roundabout mandatory":
        "Enter the roundabout according to the indicated direction.",

    "End of no passing":
        "The previous no-passing restriction ends."
}


# ============================================================
# 5. PRIORITY SYSTEM
# ============================================================

HIGH_PRIORITY = [
    "Stop",
    "Yield",
    "No entry",
    "No vehicles",
    "Speed limit (20km/h)",
    "Speed limit (30km/h)",
    "Speed limit (50km/h)",
    "Speed limit (60km/h)",
    "Speed limit (70km/h)",
    "Speed limit (80km/h)",
    "Speed limit (100km/h)",
    "Speed limit (120km/h)"
]

MEDIUM_PRIORITY = [
    "General caution",
    "Dangerous curve to the left",
    "Dangerous curve to the right",
    "Double curve",
    "Bumpy road",
    "Slippery road",
    "Road narrows on the right",
    "Road work",
    "Traffic signals",
    "Pedestrians",
    "Children crossing",
    "Bicycles crossing",
    "Beware of ice/snow",
    "Wild animals crossing"
]


def get_priority(sign_name):

    if sign_name in HIGH_PRIORITY:
        return "HIGH"

    if sign_name in MEDIUM_PRIORITY:
        return "MEDIUM"

    return "LOW"


# ============================================================
# 6. IMAGE LOCATION
# ============================================================

def get_location(x1, y1, x2, y2, width, height):

    center_x = (x1 + x2) / 2
    center_y = (y1 + y2) / 2

    if center_x < width / 3:
        horizontal = "Left"
    elif center_x < 2 * width / 3:
        horizontal = "Center"
    else:
        horizontal = "Right"

    if center_y < height / 3:
        vertical = "Upper"
    elif center_y < 2 * height / 3:
        vertical = "Middle"
    else:
        vertical = "Lower"

    return f"{horizontal} - {vertical}"


# ============================================================
# 7. HOG FEATURE EXTRACTION
# ============================================================

def extract_hog_features(crop):

    crop = cv2.resize(crop, (32, 32))

    gray = np.mean(
        crop,
        axis=2
    ).astype(np.uint8)

    features = hog(
        gray,
        orientations=9,
        pixels_per_cell=(8, 8),
        cells_per_block=(2, 2)
    )

    return features


# ============================================================
# 8. PRIORITY SCORE
# ============================================================

def priority_score(priority):

    if priority == "HIGH":
        return 3

    if priority == "MEDIUM":
        return 2

    return 1


# ============================================================
# 9. OVERALL ATTENTION LEVEL
# ============================================================

def get_attention_level(detections):

    if not detections:
        return "NO SIGN DETECTED"

    priorities = [
        d["Priority"]
        for d in detections
    ]

    if "HIGH" in priorities:
        return "HIGH ATTENTION"

    if "MEDIUM" in priorities:
        return "CAUTION"

    return "NORMAL"


# ============================================================
# 10. SCENE INSTRUCTION ENGINE
# ============================================================

def generate_scene_instruction(detections):

    if not detections:
        return (
            "No traffic sign was detected. "
            "Continue normal visual observation."
        )

    instructions = []

    # STOP
    if any(
        d["Detected Sign"] == "Stop"
        for d in detections
    ):
        instructions.append(
            "STOP: Come to a complete stop and proceed only when safe."
        )

    # Speed limits
    speed_values = []

    for d in detections:

        if "Speed limit" in d["Detected Sign"]:

            try:
                value = int(
                    d["Detected Sign"]
                    .split("(")[1]
                    .split("km/h")[0]
                )

                speed_values.append(value)

            except:
                pass

    if speed_values:

        lowest_speed = min(speed_values)

        instructions.append(
            f"Speed control: Maintain speed at or below "
            f"{lowest_speed} km/h."
        )

    # Yield
    if any(
        d["Detected Sign"] == "Yield"
        for d in detections
    ):
        instructions.append(
            "Give way to other road users before proceeding."
        )

    # Curves
    if any(
        "curve" in d["Detected Sign"].lower()
        for d in detections
    ):
        instructions.append(
            "Curve warning: Reduce speed and prepare "
            "for changing road direction."
        )

    # Road work
    if any(
        d["Detected Sign"] == "Road work"
        for d in detections
    ):
        instructions.append(
            "Road work detected: Slow down and remain "
            "alert for workers or changes in road layout."
        )

    # Slippery
    if any(
        d["Detected Sign"] == "Slippery road"
        for d in detections
    ):
        instructions.append(
            "Slippery-road warning: Avoid sudden braking "
            "or steering."
        )

    # Pedestrians / children
    if any(
        d["Detected Sign"] in [
            "Pedestrians",
            "Children crossing"
        ]
        for d in detections
    ):
        instructions.append(
            "Pedestrian warning: Reduce speed and watch "
            "carefully for people crossing."
        )

    # Overtaking
    if any(
        "No passing" in d["Detected Sign"]
        for d in detections
    ):
        instructions.append(
            "Overtaking restriction: Do not overtake."
        )

    if not instructions:

        instructions.append(
            "Follow the detected road signs and "
            "continue with appropriate caution."
        )

    return "\n".join(
        f"• {instruction}"
        for instruction in instructions
    )


# ============================================================
# 11. MAIN ROAD-SCENE ANALYSIS
# ============================================================

def recognize_road_image(input_image):

    if input_image is None:

        return (
            None,
            "Please upload a road image.",
            pd.DataFrame(),
            None
        )

    # --------------------------------------------------------
    # Convert image to OpenCV
    # --------------------------------------------------------

    image = np.array(input_image)

    if image.shape[-1] == 4:
        image = image[:, :, :3]

    image = cv2.cvtColor(
        image,
        cv2.COLOR_RGB2BGR
    )

    original = image.copy()

    height, width = image.shape[:2]

    # --------------------------------------------------------
    # YOLO DETECTION
    # --------------------------------------------------------

    yolo_results = detector.predict(
        source=image,
        conf=0.15,
        verbose=False
    )

    boxes = yolo_results[0].boxes

    detections = []

    # --------------------------------------------------------
    # PROCESS DETECTED SIGNS
    # --------------------------------------------------------

    for box in boxes:

        x1, y1, x2, y2 = (
            box.xyxy[0]
            .cpu()
            .numpy()
            .astype(int)
        )

        detection_confidence = float(
            box.conf[0]
            .cpu()
            .numpy()
        )

        x1 = max(0, x1)
        y1 = max(0, y1)
        x2 = min(width, x2)
        y2 = min(height, y2)

        if x2 <= x1 or y2 <= y1:
            continue

        # ----------------------------------------------------
        # EXPANDED CROP
        # ----------------------------------------------------

        box_width = x2 - x1
        box_height = y2 - y1

        pad_x = int(box_width * 0.10)
        pad_y = int(box_height * 0.10)

        cx1 = max(0, x1 - pad_x)
        cy1 = max(0, y1 - pad_y)
        cx2 = min(width, x2 + pad_x)
        cy2 = min(height, y2 + pad_y)

        crop = original[
            cy1:cy2,
            cx1:cx2
        ]

        if crop.size == 0:
            continue

        # ----------------------------------------------------
        # HOG
        # ----------------------------------------------------

        features = extract_hog_features(crop)

        # ----------------------------------------------------
        # SVM
        # ----------------------------------------------------

        predicted_class = int(
            classifier.predict([features])[0]
        )

        sign_name = SIGN_NAMES[predicted_class]

        category = get_category(sign_name)

        priority = get_priority(sign_name)

        action = DRIVER_ACTIONS.get(
            sign_name,
            "Proceed carefully and follow the traffic sign."
        )

        location = get_location(
            x1,
            y1,
            x2,
            y2,
            width,
            height
        )

        detections.append({

            "Class ID": predicted_class,

            "Detected Sign": sign_name,

            "Category": category,

            "Priority": priority,

            "Detection Confidence (%)":
                round(
                    detection_confidence * 100,
                    1
                ),

            "Location": location,

            "Driver Assistance": action,

            "Crop": crop,

            "Box": (x1, y1, x2, y2)
        })


    # ========================================================
    # SORT BY PRIORITY
    # ========================================================

    detections.sort(
        key=lambda d:
        -priority_score(d["Priority"])
    )


    # ========================================================
    # ANNOTATED IMAGE
    # ========================================================

    annotated = original.copy()

    for d in detections:

        x1, y1, x2, y2 = d["Box"]

        confidence = d[
            "Detection Confidence (%)"
        ]

        # Green bounding box
        cv2.rectangle(
            annotated,
            (x1, y1),
            (x2, y2),
            (0, 255, 0),
            3
        )

        label = (
            f"{d['Detected Sign']} "
            f"| {confidence:.0f}%"
        )

        font = cv2.FONT_HERSHEY_SIMPLEX

        font_scale = 0.55

        thickness = 2

        text_size, baseline = cv2.getTextSize(
            label,
            font,
            font_scale,
            thickness
        )

        text_width = text_size[0]

        text_height = text_size[1]

        label_y = max(
            y1 - 8,
            text_height + 10
        )

        # Label background
        cv2.rectangle(
            annotated,
            (
                x1,
                label_y - text_height - baseline - 5
            ),
            (
                x1 + text_width + 8,
                label_y + 5
            ),
            (0, 255, 0),
            -1
        )

        # Label text
        cv2.putText(
            annotated,
            label,
            (x1 + 3, label_y),
            font,
            font_scale,
            (0, 0, 0),
            thickness,
            cv2.LINE_AA
        )


    annotated_rgb = cv2.cvtColor(
        annotated,
        cv2.COLOR_BGR2RGB
    )


    # ========================================================
    # NO DETECTION CASE
    # ========================================================

    if not detections:

        summary = """
## Road Scene Analysis

### Detection Overview

**Traffic signs detected:** 0

**Attention Level:** NO SIGN DETECTED

No traffic sign was detected above the current
detection threshold.

Continue normal visual observation of the road.
"""

        return (
            annotated_rgb,
            summary,
            pd.DataFrame(),
            None
        )


    # ========================================================
    # SCENE STATISTICS
    # ========================================================

    total_signs = len(detections)

    high_count = sum(
        d["Priority"] == "HIGH"
        for d in detections
    )

    medium_count = sum(
        d["Priority"] == "MEDIUM"
        for d in detections
    )

    low_count = sum(
        d["Priority"] == "LOW"
        for d in detections
    )

    categories = sorted(
        set(
            d["Category"]
            for d in detections
        )
    )

    attention_level = get_attention_level(
        detections
    )


    # ========================================================
    # HIGHEST PRIORITY SIGN
    # ========================================================

    highest_priority = detections[0]


    # ========================================================
    # SCENE INSTRUCTION
    # ========================================================

    scene_instruction = generate_scene_instruction(
        detections
    )


    # ========================================================
    # SCENE SUMMARY
    # ========================================================

    summary = f"""
## Road Scene Analysis

### Detection Overview

**Traffic signs detected:** {total_signs}

**Overall attention level:** {attention_level}

**High priority:** {high_count}

**Medium priority:** {medium_count}

**Low priority:** {low_count}

**Categories present:** {", ".join(categories)}

---

### Highest-Priority Sign

**{highest_priority["Detected Sign"]}**

**Category:** {highest_priority["Category"]}

**Priority:** {highest_priority["Priority"]}

**Location:** {highest_priority["Location"]}

**Detection confidence:** {highest_priority["Detection Confidence (%)"]}%

---

### Combined Driver Assistance

{scene_instruction}

---

### Processing Pipeline

**1. YOLO11n**

Detects and localizes traffic signs in the road scene.

↓

**2. Sign Cropping**

Each detected sign is isolated from the road image.

↓

**3. HOG Feature Extraction**

Extracts visual shape and edge features.

↓

**4. SVM Classification**

Classifies the cropped sign into one of the 43 GTSRB classes.

↓

**5. Metadata Analysis**

Adds category, priority, location and driving guidance.

↓

**6. Scene-Level Interpretation**

Combines multiple detected signs into a single road-safety summary.
"""


    # ========================================================
    # RESULT TABLE
    # ========================================================

    table_data = []

    for d in detections:

        table_data.append({

            "Class ID":
                d["Class ID"],

            "Detected Sign":
                d["Detected Sign"],

            "Category":
                d["Category"],

            "Priority":
                d["Priority"],

            "Detection Confidence (%)":
                d["Detection Confidence (%)"],

            "Location":
                d["Location"],

            "Driver Assistance":
                d["Driver Assistance"]
        })


    result_table = pd.DataFrame(
        table_data
    )


    # ========================================================
    # SIGN CROP CONTACT SHEET
    # ========================================================

    crop_images = []

    for i, d in enumerate(detections):

        crop = d["Crop"].copy()

        crop = cv2.cvtColor(
            crop,
            cv2.COLOR_BGR2RGB
        )

        crop = cv2.resize(
            crop,
            (180, 180)
        )

        # White space above crop
        crop = cv2.copyMakeBorder(
            crop,
            45,
            5,
            5,
            5,
            cv2.BORDER_CONSTANT,
            value=(255, 255, 255)
        )

        title = f"Sign {i + 1}"

        cv2.putText(
            crop,
            title,
            (8, 20),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.55,
            (0, 0, 0),
            2,
            cv2.LINE_AA
        )

        sign_text = d["Detected Sign"]

        if len(sign_text) > 24:
            sign_text = sign_text[:24] + "..."

        cv2.putText(
            crop,
            sign_text,
            (8, 38),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.38,
            (0, 0, 0),
            1,
            cv2.LINE_AA
        )

        crop_images.append(crop)


    # Arrange horizontally
    if crop_images:

        crop_sheet = np.hstack(
            crop_images
        )

    else:

        crop_sheet = None


    # ========================================================
    # RETURN RESULTS
    # ========================================================

    return (
        annotated_rgb,
        summary,
        result_table,
        crop_sheet
    )


# ============================================================
# 12. GRADIO INTERFACE
# ============================================================

with gr.Blocks(
    title="Traffic Sign Recognition & Road Scene Analysis"
) as road_demo:

    gr.Markdown(
        """
# Traffic Sign Recognition & Road Scene Analysis

### YOLO11n + HOG + SVM

Upload a road-scene image containing one or more traffic signs.

The system detects signs, classifies them, analyzes their
importance and provides scene-level driver assistance.
"""
    )


    # --------------------------------------------------------
    # INPUT / OUTPUT
    # --------------------------------------------------------

    with gr.Row():

        with gr.Column():

            input_image = gr.Image(
                type="numpy",
                label="Upload Road Image"
            )

            analyze_button = gr.Button(
                "Analyze Road Scene",
                variant="primary"
            )


        with gr.Column():

            output_image = gr.Image(
                label="Detected Traffic Signs"
            )


    # --------------------------------------------------------
    # SCENE ANALYSIS
    # --------------------------------------------------------

    gr.Markdown(
        "## Road Scene Intelligence"
    )

    summary_output = gr.Markdown()


    # --------------------------------------------------------
    # TABLE
    # --------------------------------------------------------

    gr.Markdown(
        "## Detected Sign Details"
    )

    result_table = gr.Dataframe(
        interactive=False,
        wrap=True
    )


    # --------------------------------------------------------
    # CROPS
    # --------------------------------------------------------

    gr.Markdown(
        "## Individual Sign Analysis"
    )

    crop_output = gr.Image(
        label="Detected Sign Crops"
    )


    # --------------------------------------------------------
    # BUTTON ACTION
    # --------------------------------------------------------

    analyze_button.click(
        fn=recognize_road_image,

        inputs=input_image,

        outputs=[
            output_image,
            summary_output,
            result_table,
            crop_output
        ]
    )


# ============================================================
# 13. LAUNCH
# ============================================================

road_demo.launch(
    share=True
)

YOLO detector loaded successfully.
SVM classifier loaded successfully.
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://4d6a0865132430785a.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
